In [ ]:
# ── Robot connection ──────────────────────────────────────────────────────────
# Normal mode: running notebooks directly on the Reachy host → use 'localhost'.
# Remote mode: set REACHY_IP to the robot's IP address before connecting.
# REACHY_IP = "10.22.129.133"   # physical robot (remote)
REACHY_IP = "localhost"         # running on Reachy host (default)

In [1]:
import numpy as np
from reachy_sdk import ReachySDK
import cv2 as cv

In [2]:
#connect to the robot
reachy = ReachySDK(REACHY_IP) # replace 'localhost' with the actual IP address of your Reachy

In [3]:
reachy.right_camera

<Camera side="right" resolution=(640, 480, 3)>

In [4]:
reachy.right_camera.zoom_level

<ZoomLevel.INTER: 2>

In [5]:
import cv2

# Load pre-trained MobileNet SSD model
# NOTE: requires MobileNetSSD model files and physical camera — not available in simulator
net = cv2.dnn.readNetFromCaffe('MobileNetSSD_deploy.prototxt', 
                               'MobileNetSSD_deploy.caffemodel')

# Class labels from the model
CLASSES = ["background", "aeroplane", "bicycle", "bird", "boat",
           "bottle", "bus", "car", "cat", "chair", "cow", "diningtable",
           "dog", "horse", "motorbike", "person", "pottedplant", "sheep",
           "sofa", "train", "tvmonitor"]

# Start camera
while True:
    frame = reachy.right_camera.last_frame
    if frame is None:
        continue
# Convert to OpenCV format
    frame = cv2.cvtColor(np.array(frame), cv2.COLOR_RGB2BGR)
    (h, w) = frame.shape[:2]

    # Prepare input for model
    blob = cv2.dnn.blobFromImage(cv2.resize(frame, (300, 300)),
                                 0.007843, (300, 300), 127.5)
    net.setInput(blob)
    detections = net.forward()

    # Process detections
    for i in range(detections.shape[2]):
        confidence = detections[0, 0, i, 2]

        if confidence > 0.5:
            idx = int(detections[0, 0, i, 1])
            label = CLASSES[idx]
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            (startX, startY, endX, endY) = box.astype("int")

            # Draw box and label
            cv2.rectangle(frame, (startX, startY), (endX, endY), (0, 255, 0), 2)
            label_text = f"{label}: {confidence:.2f}"
            y = startY - 10 if startY - 10 > 10 else startY + 10
            cv2.putText(frame, label_text, (startX, y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

            # Example: respond to "bottle"
            if label == "bottle":
                print("Bottle detected! You can now make Reachy react.")

    # Show output
    cv2.imshow("Reachy Camera + Object Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

KeyboardInterrupt: 